# PyTorch en une soirée
### Cours condensé pour les candidats IOAI 2026

Objectif : etre capable de faire des calculs sur tensors, comprendre l'autograd, et ecrire/entrainer un petit modele avec PyTorch. On va droit au but, tout est execute directement.

Plan :
1. Tensors (creation, shape, operations)
2. Autograd (calcul automatique des gradients)
3. Construire un modele (nn.Module)
4. Boucle d'entrainement complete
5. Exercice chrono (a faire seul, 15-20 min)

Chaque section a un exemple qui tourne + un mini exercice a faire tout de suite apres, pour pas juste lire mais coder.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

print(torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device utilise:", device)

## 1. Tensors : la brique de base

Un tensor c'est un tableau multi-dimensionnel, comme un array numpy mais qui peut aller sur GPU et calculer des gradients automatiquement.

Points essentiels a retenir :
- `torch.tensor(...)` cree un tensor a partir de donnees Python
- `.shape` donne les dimensions
- `.dtype` donne le type (float32 par defaut pour le deep learning)
- les operations sont vectorisees (pas de boucle for)

In [ ]:
# creation de tensors
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

print("a:", a, "shape:", a.shape)
print("b:", b, "shape:", b.shape)

# tensors utilitaires
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
rand = torch.rand(2, 3)       # uniforme [0,1)
randn = torch.randn(2, 3)     # normale (0,1)

print(zeros)
print(rand)

In [ ]:
# operations de base : tout est vectorise, pas de boucle
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([10.0, 20.0, 30.0])

print(x + y)
print(x * y)          # multiplication element par element
print(x @ y)          # produit scalaire (dot product)

# matrices
M = torch.randn(3, 4)
N = torch.randn(4, 2)
print((M @ N).shape)   # multiplication matricielle -> (3, 2)

# reshape : tres utilise pour preparer des batchs
t = torch.arange(12)          # [0, 1, ..., 11]
print(t.reshape(3, 4))
print(t.reshape(3, 4).view(-1))   # remettre a plat

In [ ]:
# broadcasting : PyTorch etend automatiquement les dimensions compatibles
v = torch.tensor([1.0, 2.0, 3.0])          # shape (3,)
m = torch.ones(4, 3)                        # shape (4, 3)
print((m + v).shape)   # v est applique a chaque ligne de m, shape (4, 3)

# indexation comme numpy
t = torch.arange(20).reshape(4, 5)
print(t)
print(t[0])          # premiere ligne
print(t[:, 1])       # deuxieme colonne
print(t[1:3, 2:4])   # sous-bloc

### Exercice 1 (5 min)
1. Cree un tensor `x` de shape (5, 3) avec des valeurs aleatoires (`torch.randn`)
2. Calcule la moyenne de chaque colonne (`.mean(dim=0)`)
3. Calcule la somme de chaque ligne (`.sum(dim=1)`)
4. Multiplie `x` par 2 et ajoute 1, sans boucle for

In [ ]:
# ecris ta reponse ici
x = None  # TODO



## 2. Autograd : les gradients automatiques

C'est le coeur de PyTorch. Quand un tensor a `requires_grad=True`, PyTorch construit un graphe de calcul et peut calculer automatiquement les derivees avec `.backward()`.

C'est ce qui permet d'entrainer un reseau de neurones : on calcule une loss, on appelle `.backward()`, et PyTorch calcule le gradient de la loss par rapport a chaque parametre.

In [ ]:
# exemple simple : y = x^2, dy/dx = 2x
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

y.backward()          # calcule dy/dx
print("gradient dy/dx en x=3:", x.grad)   # doit etre 6.0

In [ ]:
# exemple avec plusieurs variables : loss simple de regression
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

x = torch.tensor(2.0)
y_vrai = torch.tensor(5.0)

y_pred = w * x + b
loss = (y_pred - y_vrai) ** 2   # erreur quadratique

loss.backward()
print("gradient par rapport a w:", w.grad)
print("gradient par rapport a b:", b.grad)

# IMPORTANT : les gradients s'accumulent, il faut les remettre a zero
# entre chaque etape d'entrainement (on le voit dans la boucle plus bas)

### Exercice 2 (5 min)
Avec `w = torch.tensor(2.0, requires_grad=True)`, calcule `z = 3*w**3 + 2*w`, appelle `.backward()`, et verifie que `w.grad` correspond bien a la derivee `9*w**2 + 2` evaluee en w=2 (donc 38.0).

In [ ]:
# ecris ta reponse ici
w = None  # TODO



## 3. Construire un modele avec nn.Module

En pratique on n'ecrit jamais les gradients a la main. On definit un modele en heritant de `nn.Module`, avec :
- `__init__` : on declare les couches (les "layers")
- `forward` : on decrit comment les donnees traversent le modele

Ici un petit MLP (multi-layer perceptron) pour de la classification binaire, mais la structure est la meme pour n'importe quel modele.

In [ ]:
class PetitMLP(nn.Module):
    def __init__(self, dim_entree, dim_cachee, dim_sortie):
        super().__init__()
        self.couche1 = nn.Linear(dim_entree, dim_cachee)
        self.activation = nn.ReLU()
        self.couche2 = nn.Linear(dim_cachee, dim_sortie)

    def forward(self, x):
        x = self.couche1(x)
        x = self.activation(x)
        x = self.couche2(x)
        return x

modele = PetitMLP(dim_entree=4, dim_cachee=16, dim_sortie=1).to(device)
print(modele)

# test rapide avec un batch de 8 exemples, 4 features chacun
batch_exemple = torch.randn(8, 4).to(device)
sortie = modele(batch_exemple)
print("shape de sortie:", sortie.shape)   # (8, 1)

## 4. Boucle d'entrainement complete

C'est le pattern a connaitre par coeur, on le reutilise partout :

1. `optimizer.zero_grad()` : remettre les gradients a zero
2. `y_pred = modele(x)` : forward pass
3. `loss = fonction_loss(y_pred, y_vrai)` : calculer l'erreur
4. `loss.backward()` : calculer les gradients
5. `optimizer.step()` : mettre a jour les poids

On genere des donnees synthetiques pour un probleme de classification binaire et on entraine le modele dessus.

In [ ]:
# donnees synthetiques : classification binaire simple
torch.manual_seed(0)
N = 200
X = torch.randn(N, 4)
# regle arbitraire : classe 1 si somme des features > 0
y_vrai = (X.sum(dim=1) > 0).float().unsqueeze(1)

X, y_vrai = X.to(device), y_vrai.to(device)

modele = PetitMLP(dim_entree=4, dim_cachee=16, dim_sortie=1).to(device)
fonction_loss = nn.BCEWithLogitsLoss()   # loss adaptee a la classification binaire
optimizer = optim.Adam(modele.parameters(), lr=0.01)

nb_epochs = 100
for epoch in range(nb_epochs):
    optimizer.zero_grad()
    y_pred = modele(X)
    loss = fonction_loss(y_pred, y_vrai)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f}")

print("loss finale:", loss.item())

In [ ]:
# evaluer la precision du modele entraine
with torch.no_grad():
    y_pred = modele(X)
    y_pred_classe = (torch.sigmoid(y_pred) > 0.5).float()
    precision = (y_pred_classe == y_vrai).float().mean()
    print("precision sur les donnees d'entrainement:", precision.item())

### Points a retenir (cheatsheet rapide)

- `requires_grad=True` -> PyTorch suit les operations pour calculer les gradients
- `optimizer.zero_grad()` avant chaque `backward()`, sinon les gradients s'accumulent
- `with torch.no_grad():` quand on evalue, pour ne pas gaspiller de memoire sur les gradients
- `nn.Linear(in, out)` = une couche entierement connectee
- Choix de loss : `nn.MSELoss()` pour la regression, `nn.BCEWithLogitsLoss()` pour classification binaire, `nn.CrossEntropyLoss()` pour classification multi-classe
- `optim.Adam(modele.parameters(), lr=...)` est un bon choix par defaut

## 6. Memotechniques pour retenir vite

### La boucle d'entrainement : "ZFLBS"
Zero grad, Forward, Loss, Backward, Step. Phrase pour retenir l'ordre : **"Zorro Fonce, Lance sa Botte Secrete"**.
- **Z**ero grad -> `optimizer.zero_grad()`
- **F**orward -> `y_pred = modele(x)`
- **L**oss -> `loss = fonction_loss(y_pred, y_vrai)`
- **B**ackward -> `loss.backward()`
- **S**tep -> `optimizer.step()`

Si l'ordre est bon mais que la loss ne descend pas, 90% du temps c'est un `zero_grad()` oublie ou un `lr` mal choisi.

### Choisir sa loss : "RBC"
- **R**egression -> `MSELoss`
- **B**inaire (2 classes) -> `BCEWithLogitsLoss`
- **C**lasses multiples -> `CrossEntropyLoss`

### Broadcasting en une phrase
"Aligne a droite, complete avec des 1." Les shapes se comparent de la droite vers la gauche, une dimension de 1 s'etire pour matcher l'autre.

### `.backward()` vs `torch.no_grad()`
- On entraine -> les gradients sont utiles -> pas de `no_grad`
- On evalue / on predit seulement -> pas besoin de gradients -> toujours `with torch.no_grad():`
Phrase : "j'apprends je garde la trace, j'utilise je n'ai pas besoin de la trace".

## 7. Strategie pour progresser vite (pas juste PyTorch)

Vous avez peu de temps avant l'epreuve, la vitesse d'assimilation compte autant que la comprehension. Quelques principes qui marchent vraiment :

1. **Rappel actif plutot que relecture** : apres chaque section de ce notebook, fermez-le et reecrivez le code de memoire dans une cellule vide. Se tromper et se corriger fait plus progresser que relire.
2. **Construire tout de suite, pas juste comprendre** : ne restez jamais plus de 10 minutes sur de la theorie sans coder un exemple derriere. Un concept qui n'a pas ete tape au clavier n'est pas acquis.
3. **Expliquer a voix haute (technique Feynman)** : si vous ne pouvez pas expliquer `autograd` a votre binome en 2 phrases simples, c'est que ce n'est pas encore clair, retournez a la section 2.
4. **Repetition espacee sur les erreurs, pas sur tout** : gardez une liste courte des 5 erreurs que vous faites le plus souvent (ex: oublier `zero_grad`, mauvaise shape) et relisez cette liste chaque matin, pas tout le cours.
5. **Lire le message d'erreur en entier avant de chercher sur internet** : les erreurs PyTorch de shape mismatch disent presque toujours exactement quelles dimensions ne matchent pas.
6. **Un seul concept nouveau a la fois en condition d'epreuve** : ne pas melanger apprentissage d'un nouveau concept ET debug d'un vieux bug le meme jour avant l'epreuve.

### Ressources pour approfondir vite (classees par urgence)

Pour ce soir, si le temps manque, dans cet ordre de priorite :
1. **PyTorch official "60 minute blitz"** : https://docs.pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html (le strict minimum vital, se fait vraiment en moins d'une heure)
2. **d2l.ai (Dive into Deep Learning)** : https://d2l.ai/ (chapitres sur les fondamentaux, exemples PyTorch executables, tres bon pour revoir la theorie derriere chaque bloc de code)
3. **Documentation officielle nn.Module** : https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html (a garder ouvert en onglet pendant les exercices)

Si plus de temps disponible dans les prochains jours :
4. **fast.ai - Practical Deep Learning** : https://course.fast.ai/ (approche tres pratique, va droit au but sur les projets)
5. **Videos Andrej Karpathy "Neural Networks: Zero to Hero"** : https://karpathy.ai/zero-to-hero.html (pour comprendre l'autograd en profondeur, il recode un mini-autograd a la main, tres formateur)

Pour l'IOAI specifiquement, les ressources officielles a connaitre par coeur :
- Syllabus officiel 2026 : https://ioai-official.org/republic-of-kazakhstan/syllabus-2026/
- Regles du concours 2026 (PDF) : https://ioai-official.org/wp-content/uploads/2026/06/IOAI2026-Contest-Rules-and-Tehnical-Appendix.pdf
- Repo GitHub des 3 taches At-Home 2026 : https://github.com/IOAI-official/IOAI-2026
- Taches Kaggle At-Home : https://www.kaggle.com/competitions/ioai-2026-home-task-1 et https://www.kaggle.com/competitions/ioai-2026-home-task-2

## 5. Exercice final chrono (15-20 min)

A faire seul, sans aide, comme en condition d'epreuve.

Consignes :
1. Genere des donnees synthetiques pour une regression : `X` de shape (100, 1), et `y = 3*X + 2 + bruit` (ajoute un peu de bruit gaussien avec `torch.randn`)
2. Definis un modele avec une seule couche `nn.Linear(1, 1)`
3. Entraine-le avec `nn.MSELoss()` et `optim.SGD` ou `optim.Adam`, pendant 200 epochs
4. Affiche la loss toutes les 50 epochs
5. Bonus : affiche les valeurs apprises de poids et biais (`modele.weight`, `modele.bias`), ca doit se rapprocher de 3 et 2

In [ ]:
# ecris ta solution ici



